In [64]:
import os
import re
import sys
import nltk
import pandas as pd
from string import punctuation
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sys.path.append(os.path.abspath('../scripts'))

from image import processing

nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/Damir/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [65]:
df = pd.read_csv('../data/selected.csv', index_col=[0])

In [66]:
words = df['artist_description'].str.lower().str.replace(rf'[{re.escape(punctuation)}]', '', regex=True).str.split()
df['artist_description_words_count'] = words.str.len()
df['artist_description_mean_word_length'] = words.explode().str.len().groupby(level=0).mean()

In [67]:
words = df['song_description'].str.lower().str.replace(rf'[{re.escape(punctuation)}]', '', regex=True).str.split()
df['song_description_words_count'] = words.str.len()
df['song_description_mean_word_length'] = words.explode().str.len().groupby(level=0).mean()

In [ ]:
text = df['lyrics'].str.lower().str.replace(rf'[{re.escape(punctuation)}]', '', regex=True)
words = text.str.split()
df['lyrics_words_count'] = words.str.len()
df['unique_lyrics_words_count'] = words.explode().groupby(level=0).apply(lambda x: x.drop_duplicates().size)
df['lyrics_mean_word_length'] = words.explode().str.len().groupby(level=0).mean()
df['chorus_count'] = df['lyrics'].str.count(r'\[Chorus\]')
df['lyrics_unique_ratio'] = df['unique_lyrics_words_count'] / df['lyrics_words_count']
df['lyrics_sentiment'] = df['lyrics'].apply(lambda x: None if pd.isna(x) else sia.polarity_scores(x)['compound'])

In [ ]:
image_features = df['image_url'].dropna()[:1000].apply(processing)

df = df.join(image_features)

ValueError: columns overlap but no suffix specified: Index(['url', 'light', 'contrast', 'saturation', 'hue', 'clarity', 'mean_r',
       'mean_g', 'mean_b'],
      dtype='str')

In [ ]:
df.to_csv('../data/engineered.csv')